# Quickstart

`point-collocation` gets matchups to lat/lon using the pixel center that is closest to the lat/lon point (equivalent to method="nearest"). For time, you can select a buffer of 0, which means the time of the point must be within the time range of the file or a buffer like buffer="1D" to find files within 1 day of the point. Using a buffer can help for L2 files with short windows (minutes) or collections with infrequent files.

* Create a plan for files to use `pc.plan()`
* Print the plan to check it `plan.summary()`
* Do the plan and get matchups for variables `pc.matchup(plan, geometry='grid', variables=['var'])`

## Prerequisite -- Login to EarthData

The examples here use NASA EarthData and you need to have an account with EarthData. Make sure you can login.

In [6]:
import earthaccess
earthaccess.login()

## Get some points to matchup

In [1]:
from pathlib import Path
import pandas as pd

HERE = Path.cwd()
POINTS_CSV = HERE / "fixtures" / "points.csv"
df_points = pd.read_csv(POINTS_CSV)  # lat, lon, date columns
print(len(df_points))
df_points.head()

595


,lat,lon,date
0,27.3835,-82.7375,2024-06-13
1,27.1190,-82.7125,2024-06-14
2,26.9435,-82.8170,2024-06-14
3,26.6875,-82.8065,2024-06-14
4,26.6675,-82.6455,2024-06-14


## Start plan -- Take a look at the files in a collection

Now we use the point_collocation package. First we will look at the files available and figure out which ones we want.

In [9]:
%%time
import point_collocation as pc
plan = pc.plan(
    df_points,
    data_source="earthaccess",
    source_kwargs={
        "short_name": "PACE_OCI_L3M_RRS",
    }
)

CPU times: user 2.83 s, sys: 0 ns, total: 2.83 s
Wall time: 7.34 s


In [10]:
plan.summary(n=1)

Plan: 595 points → 464 unique granule(s)
  Points with 0 matches : 0
  Points with >1 matches: 595
  Variables  : []
  Time buffer: 0 days 00:00:00

First 1 point(s):
  [0] lat=27.3835, lon=-82.7375, time=2024-06-13 12:00:00: 16 match(es)
    → https://obdaac-tea.earthdatacloud.nasa.gov/ob-cumulus-prod-public/PACE_OCI.20240321_20240620.L3m.SNSP.RRS.V3_1.Rrs.0p1deg.nc
    → https://obdaac-tea.earthdatacloud.nasa.gov/ob-cumulus-prod-public/PACE_OCI.20240321_20240620.L3m.SNSP.RRS.V3_1.Rrs.4km.nc
    → https://obdaac-tea.earthdatacloud.nasa.gov/ob-cumulus-prod-public/PACE_OCI.20240516_20240616.L3m.R32.RRS.V3_1.Rrs.0p1deg.nc
    → https://obdaac-tea.earthdatacloud.nasa.gov/ob-cumulus-prod-public/PACE_OCI.20240516_20240616.L3m.R32.RRS.V3_1.Rrs.4km.nc
    → https://obdaac-tea.earthdatacloud.nasa.gov/ob-cumulus-prod-public/PACE_OCI.20240524_20240624.L3m.R32.RRS.V3_1.Rrs.0p1deg.nc
    → https://obdaac-tea.earthdatacloud.nasa.gov/ob-cumulus-prod-public/PACE_OCI.20240524_20240624.L3m.R32.RRS.V3_1

## Create new plan with filter on file names

We will use the monthly 4km files.

In [2]:
%%time
import point_collocation as pc
plan = pc.plan(
    df_points,
    data_source="earthaccess",
    source_kwargs={
        "short_name": "PACE_OCI_L3M_RRS",
        "granule_name": "*.MO.*.4km.*",
    }
)

CPU times: user 650 ms, sys: 58.9 ms, total: 709 ms
Wall time: 21.8 s


In [13]:
# check the plan and see how many files per point
# we want 1 file per point in this case
# Looks like 6 monthly files
plan.summary()

Plan: 595 points → 6 unique granule(s)
  Points with 0 matches : 0
  Points with >1 matches: 0
  Variables  : []
  Time buffer: 0 days 00:00:00

First 5 point(s):
  [0] lat=27.3835, lon=-82.7375, time=2024-06-13 12:00:00: 1 match(es)
    → https://obdaac-tea.earthdatacloud.nasa.gov/ob-cumulus-prod-public/PACE_OCI.20240601_20240630.L3m.MO.RRS.V3_1.Rrs.4km.nc
  [1] lat=27.1190, lon=-82.7125, time=2024-06-14 12:00:00: 1 match(es)
    → https://obdaac-tea.earthdatacloud.nasa.gov/ob-cumulus-prod-public/PACE_OCI.20240601_20240630.L3m.MO.RRS.V3_1.Rrs.4km.nc
  [2] lat=26.9435, lon=-82.8170, time=2024-06-14 12:00:00: 1 match(es)
    → https://obdaac-tea.earthdatacloud.nasa.gov/ob-cumulus-prod-public/PACE_OCI.20240601_20240630.L3m.MO.RRS.V3_1.Rrs.4km.nc
  [3] lat=26.6875, lon=-82.8065, time=2024-06-14 12:00:00: 1 match(es)
    → https://obdaac-tea.earthdatacloud.nasa.gov/ob-cumulus-prod-public/PACE_OCI.20240601_20240630.L3m.MO.RRS.V3_1.Rrs.4km.nc
  [4] lat=26.6675, lon=-82.6455, time=2024-06-14 

## Check the variables in the files

This will open one file and show us the variables. We want 'Rrs' in this case.

In [14]:
plan.show_variables(geometry="grid")

Dimensions : {'lat': 4320, 'lon': 8640, 'wavelength': 172, 'rgb': 3, 'eightbitcolor': 256}
Variables  : ['Rrs', 'palette']


## Get the matchups using our plan

Let's start with 100 points since 595 might take awhile.

In [15]:
%%time
res = pc.matchup(plan[0:100], geometry="grid", variables=["Rrs"])

CPU times: user 12.1 s, sys: 1.06 s, total: 13.2 s
Wall time: 18.8 s


In [16]:
res.head()

,lat,lon,time,granule_id,Rrs_346,Rrs_348,Rrs_351,Rrs_353,Rrs_356,Rrs_358,...,Rrs_706,Rrs_707,Rrs_708,Rrs_709,Rrs_711,Rrs_712,Rrs_713,Rrs_714,Rrs_717,Rrs_719
0,27.3835,-82.7375,2024-06-13 12:00:00,https://obdaac-tea.earthdatacloud.nasa.gov/ob-...,0.004034,0.004070,0.004170,0.004278,0.004462,0.004604,...,0.000224,0.000202,0.000190,0.000176,0.000168,0.000156,0.000144,0.000134,0.000158,0.000202
1,27.1190,-82.7125,2024-06-14 12:00:00,https://obdaac-tea.earthdatacloud.nasa.gov/ob-...,0.004562,0.004616,0.004700,0.004692,0.004806,0.005070,...,0.000108,0.000094,0.000084,0.000078,0.000072,0.000066,0.000060,0.000048,0.000062,0.000098
2,26.9435,-82.8170,2024-06-14 12:00:00,https://obdaac-tea.earthdatacloud.nasa.gov/ob-...,0.005112,0.005282,0.005458,0.005582,0.005868,0.006226,...,0.000118,0.000108,0.000102,0.000098,0.000098,0.000092,0.000086,0.000068,0.000052,0.000066
3,26.6875,-82.8065,2024-06-14 12:00:00,https://obdaac-tea.earthdatacloud.nasa.gov/ob-...,0.004648,0.004904,0.005108,0.005242,0.005548,0.005944,...,0.000178,0.000158,0.000148,0.000138,0.000130,0.000126,0.000126,0.000120,0.000158,0.000230
4,26.6675,-82.6455,2024-06-14 12:00:00,https://obdaac-tea.earthdatacloud.nasa.gov/ob-...,0.004944,0.005064,0.005190,0.005288,0.005504,0.005838,...,0.000094,0.000078,0.000068,0.000062,0.000058,0.000054,0.000052,0.000050,0.000106,0.000166


In [3]:
%%time
res = pc.matchup(plan, geometry="grid", variables=["Rrs"])

CPU times: user 1min 3s, sys: 2.75 s, total: 1min 6s
Wall time: 1min 15s
